## Transaction Detection

**Information Need:** Identify groups of events that potentially belong to the same short- or long-running transaction within the same case.

**Motivation:** Events belonging to the same transaction may be recorded as separate activity types and may occur repeatedly within a case. Identifying activities with similar case-level occurrence helps analysts recognize candidate transaction boundaries, understand which recorded events may represent a common unit of work, and reduce behavioral complexity in subsequent analyses.

**Precondition:**  Define a similarity threshold, a minimum transaction-set size, and any activity types to exclude.

**Approach:** Characterize how activity types co-occur within cases and identify sets of activities with sufficiently similar co-occurrence as candidate transactions. Rank the resulting candidate sets according to the strength of their observed association.
For example, construct a case–activity count table and represent each activity type as a count vector across cases. Compute a similarity score for every pair of activity vectors. Create a graph connecting activity pairs whose similarity meets the specified threshold, then extract maximal cliques in which every activity pair satisfies the threshold. Rank the resulting candidates by set size and mean pairwise similarity.

**Output:** A ranked collection of candidate transaction groups, each containing a set of activity types, together with its set size and  similarity measure.

### Proposed solution

- Create a case-event count table (rows = cases, columns = events). (cf. Case-wise Activity Occurrence pattern)
- Exclude optional events if needed.
- Represent each event as a count vector across cases.
- Compute pairwise **weighted Jaccard similarity** between event vectors:

$$
\mathrm{sim}(x,y)=\frac{\sum_k \min(x_k,y_k)}{\sum_k \max(x_k,y_k)}
$$

- Build a graph where events are nodes and an edge exists when similarity is `>= SIM_THRESHOLD`.
- Extract maximal cliques to obtain strict transaction candidates (all pairs in a clique satisfy the threshold).
- Keep cliques with size `>= MIN_SET_SIZE` and rank them using pairwise similarity statistics.

In [ ]:
import networkx as nx
import numpy as np
import pandas as pd
import pm4py

# --- Configuration -----------------------------------------------------------

LOG_PATH = "../../data/Road_Traffic_Fine_Management_Process.xes"

CASE_ID = "case:concept:name"
ACTIVITY = "concept:name"
TIMESTAMP = "time:timestamp"

# Events to ignore.
EXCLUDED_EVENTS = []

# Minimum number of events required for a candidate transaction set.
MIN_SET_SIZE = 2

# Similarity cutoff for connecting two events in the graph.
# If similarity(event_i, event_j) >= SIM_THRESHOLD, an edge is added.
SIM_THRESHOLD = 0.90

print(f'Excluded events: {EXCLUDED_EVENTS}')
print(f"Sim Threshold: {SIM_THRESHOLD}")
print(f'Min set size: {MIN_SET_SIZE}')

In [ ]:
event_log = pm4py.read_xes(LOG_PATH)

#ensure ordering
event_log = event_log.sort_values([CASE_ID, TIMESTAMP]).reset_index(drop=True)

print(f'Loaded events: {len(event_log):,}')
print(f"Cases: {event_log[CASE_ID].nunique():,} | Activities: {event_log[ACTIVITY].nunique():,}")
display(event_log.head())

### Execute Case-wise Activity Occurrence Pattern

In [ ]:
# Build case-event count table and exclude selected events
count_matrix = (
    event_log.groupby([CASE_ID, ACTIVITY]).size()
    .rename('count').reset_index()
    .pivot(index=CASE_ID, columns=ACTIVITY, values='count')
    .fillna(0).astype(int)
)

# Exclude selected events
use_cols = [c for c in count_matrix.columns if c not in EXCLUDED_EVENTS]
count_matrix = count_matrix[use_cols].copy()

#Display table
display(count_matrix.head())


### Pattern execution

#### Pairwise similarity transaction sets

1. Build event count vectors.
2. Compute pairwise **weighted Jaccard similarity** between events.
3. Build a threshold graph from similarity values.
4. Extract maximal cliques and keep sets of size >= `MIN_SET_SIZE`.

Strict rule used here: all events in a transaction set must be mutually similar above the threshold.


In [ ]:
# Build event count vectors from the case-event count matrix.
# Each row is one event profile across all cases.
event_vectors = count_matrix.T.astype(float).copy()

print(f"Events: {event_vectors.shape[0]} | Cases: {event_vectors.shape[1]}")
display(event_vectors.head())

#### Pairwise weighted Jaccard similarity

Compute similarity between every pair of events using count vectors:

$$
\mathrm{sim}(x,y)=
\frac{\sum_k \min(x_k,y_k)}
     {\sum_k \max(x_k,y_k)}
$$

Interpretation:
- `1.0` means identical count profile across cases.
- `0.0` means no overlap in counts across cases.


##### Small worked example (4 traces, 2 events)

To make the formula concrete, consider two event count vectors across 4 traces:
- Event A: $x = [1,\,0,\,2,\,1]$
- Event B: $y = [1,\,1,\,1,\,0]$

We compare the vectors trace by trace:
- Overlap part (minimum per trace): $\min(x,y) = [1,\,0,\,1,\,0]$
- Union part (maximum per trace): $\max(x,y) = [1,\,1,\,2,\,1]$

Now sum both parts:
- Numerator: $\sum_k \min(x_k,y_k) = 1+0+1+0 = 2$
- Denominator: $\sum_k \max(x_k,y_k) = 1+1+2+1 = 5$

So the weighted Jaccard similarity is:

$$
\mathrm{sim}(x,y)=\frac{2}{5}=0.4
$$

Reasoning: the two events share some behavior (same non-zero counts in part of the traces), but they also differ in multiple traces. The score $0.4$ reflects **partial overlap**: not dissimilar enough to be 0, and far from identical (which would be 1).

In [ ]:
# Compute pairwise weighted Jaccard similarity matrix (events x events).
X = event_vectors.to_numpy()
min_sum = np.minimum(X[:, None, :], X[None, :, :]).sum(axis=2)
max_sum = np.maximum(X[:, None, :], X[None, :, :]).sum(axis=2)

# np.divide computes element-wise min_sum / max_sum; using 'where' skips zero denominators,
# and 'out' pre-fills those skipped positions with 1.0 (for identical all-zero vector pairs).
sim_values = np.divide(
    min_sum,
    max_sum,
    out=np.ones_like(min_sum, dtype=float),
    where=max_sum > 0
    )

similarity_matrix = pd.DataFrame(
    sim_values,
    index=event_vectors.index,
    columns=event_vectors.index
    )


print(f"Similarity matrix shape: {similarity_matrix.shape}")
print(f"Value range: [{similarity_matrix.values.min():.4f}, {similarity_matrix.values.max():.4f}]")

similarity_matrix

#### Build threshold-based transaction sets (graph + cliques)

Build a graph where:
- each event is a node
- an edge exists if pairwise similarity is `>= SIM_THRESHOLD`

Then extract maximal cliques.
Each clique guarantees that every pair inside the set satisfies the threshold.

In [ ]:
# Build a boolean adjacency matrix from the similarity threshold.
# True means two events are connected (similar enough).
adjacency = (similarity_matrix >= SIM_THRESHOLD)

# Remove self-links on the diagonal (an event should not form an edge with itself).
values = adjacency.to_numpy(copy=True)
np.fill_diagonal(values, False)
adjacency = pd.DataFrame(values, index=adjacency.index, columns=adjacency.columns)


adjacency

In [ ]:
# Create an undirected graph from the adjacency matrix.
G = nx.from_pandas_adjacency(adjacency.astype(int))

# Find maximal cliques: fully connected groups of events.
# Keep only cliques that meet the minimum set size.
cliques = [sorted(list(c)) for c in nx.find_cliques(G) if len(c) >= MIN_SET_SIZE]

# Convert cliques to a candidate-set table.
candidate_sets_soft = pd.DataFrame({'event_set': cliques})
candidate_sets_soft['set_size'] = candidate_sets_soft['event_set'].map(len)

# For one candidate set, compute quality from the pairwise similarity submatrix:
# 1) take only rows/cols of events in the set
# 2) keep upper-triangle pairs (i < j) to avoid duplicates and diagonal
# 3) summarize with min and mean pairwise similarity
def set_similarity(events):
    sub = similarity_matrix.loc[events, events].values
    pair_vals = sub[np.triu_indices(len(events), k=1)]
    return float(pair_vals.mean())

candidate_sets_soft['similarity'] = candidate_sets_soft['event_set'].apply(set_similarity)

# Sort best candidates first (bigger sets, then higher similarity).
candidate_sets_soft = candidate_sets_soft.sort_values(
    ['set_size', 'similarity'],
    ascending=[False, False]
).reset_index(drop=True)

print(f"Graph nodes (events): {G.number_of_nodes()}")
print(f"Graph edges (similarity >= {SIM_THRESHOLD}): {G.number_of_edges()}")
print(f"Candidate cliques kept (size >= {MIN_SET_SIZE}): {len(candidate_sets_soft)}")
candidate_sets_soft